# Begriffe, Input-Parameter und Output-Kennzahlen auf der Website pflegen

Dieses How-to erzeugt beide Kataloge aus dem aktiven Schema mit denselben Funktionen wie der Website-Build.
Es liest keine Projektwerte und verändert weder Schema noch Excel-Dateien. Voraussetzung:
`python -m pip install -e ".[publication]"` im geklonten Repository.

## 1. Aktive Definitionen lesen

In [1]:
from pathlib import Path
import sys
import pandas as pd
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/pexl").is_dir())
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "src"))
from output_catalog import catalog_variables, build
from pexl.schema.current import SCHEMA_VERSION
kataloge = {quelle: catalog_variables(quelle) for quelle in ("IN", "OUT")}
print("Schema:", SCHEMA_VERSION)
for quelle, variablen in kataloge.items():
    print(quelle, len(variablen), "Variablen;", sum(bool(m.icon) for m in variablen), "mit Schema-Icon")
pd.DataFrame([{k: getattr(m, k) for k in ("source", "var_name", "icon", "label_de", "unit", "domain", "measure", "entity_group")}
              for variablen in kataloge.values() for m in variablen]).groupby("source").head(5)

Schema: 1_13_1_dev
IN 692 Variablen; 691 mit Schema-Icon
OUT 512 Variablen; 332 mit Schema-Icon


,source,var_name,icon,label_de,unit,domain,measure,entity_group
0,IN,tool_version,🔧,Tool Version,text,tool,version,None
1,IN,input_version_number,🔧,Inputdatenstruktur Version,text,tool,input version,None
2,IN,cfd_fPE,🔧,Primärenergiefaktor,kWhPE/kWhEE,declaration,fPE,None
3,IN,cfd_A,🔧,Ausnutzungsfaktor Erneuerbare,kWhEE/m²GF,declaration,A,None
4,IN,cfd_dx,🔧,Skalierungsfaktor,-,declaration,dx,None
692,OUT,Ta_annual_avg,None,Außentemperatur Jahresmittel,°C,weather,Wetter,None
693,OUT,Ta_avg_heating_period,None,Mittelaußentemperatur Heizperiode,°C,weather,temperature,None
694,OUT,Ta_avg_cooling_period,None,Mittelaußentemperatur Kühlperiode,°C,weather,temperature,None
695,OUT,UED_plugloads,🔌,Nutzerstrom,kWh/m²NGFa,useful_energy,plugloads,None
696,OUT,UED_lighting,💡,Beleuchtung,kWh/m²NGFa,useful_energy,lighting,None


## 2. Fehlende Metadaten sichtbar halten

Nicht hinterlegte Einheiten sind nicht automatisch dimensionslos. Deutsche Labels sind von zusätzlichen
Beschreibungen im Feld `comment` zu unterscheiden. Icons stammen ausschließlich aus `icon` im Schema;
fehlende Icons werden nicht erfunden. Die Bezeichnung bleibt auch ohne Emoji verständlich.

In [2]:
pd.DataFrame({quelle: {feld: sum(not getattr(m, feld) for m in variablen)
                                 for feld in ("label_de", "unit", "domain", "entity_group", "comment", "icon")}
              for quelle, variablen in kataloge.items()}).rename_axis("Nicht hinterlegt")

,IN,OUT
Nicht hinterlegt,,
label_de,1,16
unit,114,36
domain,12,70
entity_group,256,373
comment,582,512
icon,1,180


## 3. Beide Quarto-Fragmente mit derselben Funktion erzeugen

Für einen isolierten Probelauf wird in `local_exports` geschrieben. `build(source="IN")` bzw.
`build(source="OUT")` aktualisiert `website/assets/in-catalog.qmd` bzw. `out-catalog.qmd`.
Der vorhandene `website_assets.build` erzeugt ebenfalls beide Fragmente. Es gibt keine zweite Registry.

In [3]:
for quelle, variablen in kataloge.items():
    prefix = quelle.lower()
    ziel = ROOT / f"notebooks/local_exports/output_catalog/{prefix}-catalog.qmd"
    fragment = build(ziel, source=quelle)
    inhalt = fragment.read_text(encoding="utf-8")
    assert inhalt.count('<th scope="row">') == len(variablen)
    assert all(f'id="{prefix}-{m.var_name}"' in inhalt for m in variablen)
    assert inhalt.count('class="catalog-icon"') == sum(bool(m.icon) for m in variablen)
    print(quelle, "vollständig mit vorhandenen Schema-Icons ausgegeben.")

IN vollständig mit vorhandenen Schema-Icons ausgegeben.
OUT vollständig mit vorhandenen Schema-Icons ausgegeben.


## 4. Website aktualisieren und prüfen

„Nachschlagen“ ist ein eigener Navigationspunkt nach „Deklarieren“. Die Seite enthält Begriffe,
Input-Parameter und Output-Kennzahlen. `glossary.qmd` bindet beide Fragmente ein;
`output-catalog.js` initialisiert Suche und Filter für jeden Katalog separat.

```powershell
python scripts/output_catalog.py
quarto render website
python -m http.server 8765 --bind 127.0.0.1 --directory website/_site
```

Unter `http://127.0.0.1:8765/glossary.html#parameterkatalog` einen Input mit Icon suchen,
Domäne und Entity Group filtern und Metadaten öffnen. Danach bei `#kennzahlenkatalog` Kosten,
Flexibilität und Speicher prüfen. Änderungen eines Filters dürfen den anderen Katalog nicht verändern.
Direktlinks beginnen mit `#in-` oder `#out-`; `#out-Batt_FEC` bleibt erhalten.
Ohne JavaScript bleiben alle Domänen manuell aufklappbar.

Tests: `python -m pytest tests/test_output_catalog.py tests/test_website.py tests/test_tutorials.py`.
Die englische Website ist ein separates Unterprojekt; ein lokaler deutscher Build veröffentlicht nichts.